In [ ]:
import pandas as pd
import os

import pickle
from pathlib import Path

In [ ]:
#menigitis CASES

out_file2 = Path('/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/nonseasonal_cohort_data_table.pkl')
# 4) Reload with one line:
with open(out_file2, 'rb') as f:
    ns_df = pickle.load(f)

men = ns_df[(433403, 'Viral meningitis')]

to_drop = ['American Indian or Alaska Native']

# 2) Keep only rows whose updated_race is not in that list
men = men[~men['updated_race'].isin(to_drop)]

In [ ]:
 #adding age based on date_of_birth column
men['age'] = men['first_diag_date'].dt.year - men['date_of_birth'].dt.year
men

In [ ]:
#controls

controls = '/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-09-30_viral_disease_control_cohort_creation/Menigitis_Controls.csv'
control = pd.read_csv(controls)
control

In [ ]:
#getting age for control cohorts based on diagnosis date median date

import pandas as pd
import numpy as np

# ==== EDIT to your schema ====
ID_CASES     = "person_id"   # in cases_df
CASE_DX_DATE = "first_diag_date"          # diagnosis date (YYYY-MM-DD) in cases_df
CASE_AGE_COL = "age"        # numeric age at FIRST diagnosis in cases_df

ID_CONTROLS  = "person_id"   # in controls_df
CTRL_DOB_COL = "date_of_birth"              # date of birth (YYYY-MM-DD) in controls_df
# =============================

cases = men.copy()
ctrls = control.copy()

# 1) First diagnosis per case, then the cohort-wide median of those dates
cases[CASE_DX_DATE] = pd.to_datetime(cases[CASE_DX_DATE], errors="coerce")
first_dx = (cases
            .dropna(subset=[CASE_DX_DATE])
            .sort_values(CASE_DX_DATE)
            .groupby(ID_CASES, as_index=False)[CASE_DX_DATE].first())

index_date = first_dx[CASE_DX_DATE].median()  # single anchor date
print("Index date (median first diagnosis):", index_date.date())

# 2) Controls: age at the index date (from DOB)
ctrls[CTRL_DOB_COL] = pd.to_datetime(ctrls[CTRL_DOB_COL], errors="coerce")
ctrls["age"] = (index_date - ctrls[CTRL_DOB_COL]).dt.days / 365.25

# 3) Cases: use age-at-first-diagnosis you already have
cases["age"] = pd.to_numeric(cases[CASE_AGE_COL], errors="coerce")

# 4) Clean implausible ages
for df in (cases, ctrls):
    df.loc[(df["age"] <= 0) | (df["age"] > 120), "age"] = np.nan


In [ ]:
#W/O PCs

import pandas as pd
import numpy as np
import re

# ==== EDIT to your schema ====
ID_CASES      = "person_id"   # in cases  (these should match what you used earlier)
ID_CONTROLS   = "person_id"   # in controls
SEX_CASES     = "sex_at_birth"     # sex in cases (M/F or 1/2)
SEX_CONTROLS  = "sex_at_birth"     # sex in controls
# cases and ctrls should already have an 'age' column from your earlier step
# ======================================

# 0) Make working copies
cases = cases.copy()
ctrls = ctrls.copy()

# 1) **Harmonize ID dtypes** (critical to avoid merge error)
for df, col in ((cases, ID_CASES), (ctrls, ID_CONTROLS)):
    df[col] = df[col].astype("string")

# 2) Build base sample list (union of IDs; FID=IID, both strings)
case_ids = pd.Series(cases[ID_CASES].dropna().astype("string").unique(), dtype="string", name="IID")
ctrl_ids = pd.Series(ctrls[ID_CONTROLS].dropna().astype("string").unique(), dtype="string", name="IID")

base = pd.DataFrame({"IID": pd.concat([case_ids, ctrl_ids], ignore_index=True).drop_duplicates()})
base["FID"] = base["IID"]
base = base.astype({"FID":"string","IID":"string"})[["FID","IID"]]

# 3) Phenotype file (binary)
pheno = base.copy()
case_set, ctrl_set = set(case_ids), set(ctrl_ids)
pheno["meningitis"] = np.where(
    pheno["IID"].isin(case_set), 1,
    np.where(pheno["IID"].isin(ctrl_set), 0, np.nan)
)
pheno.to_csv("pheno.tsv", sep="\t", index=False)

# 4) Covariate file (sex + age only)
dem_cases = cases[[ID_CASES, SEX_CASES, "age"]].rename(columns={ID_CASES:"IID", SEX_CASES:"sex"})
dem_ctrls = ctrls[[ID_CONTROLS, SEX_CONTROLS, "age"]].rename(columns={ID_CONTROLS:"IID", SEX_CONTROLS:"sex"})

# force IID to string here too (pre-merge)
dem_cases["IID"] = dem_cases["IID"].astype("string")
dem_ctrls["IID"] = dem_ctrls["IID"].astype("string")

covar = pd.concat([dem_cases, dem_ctrls], ignore_index=True).drop_duplicates(subset=["IID"])
covar["FID"] = covar["IID"]

# normalize sex to M/F (keep NA if unknown)
covar["sex"] = (covar["sex"].astype(str).str.strip().str.upper()
                .replace({"MALE":"M","FEMALE":"F","1":"M","2":"F","0":np.nan,"UNKNOWN":np.nan}))
# numeric age
covar["age"] = pd.to_numeric(covar["age"], errors="coerce")

# ensure FID/IID are strings before merge
covar = covar.astype({"FID":"string","IID":"string"})

# align to base (same universe/order) and write
covar = base.merge(covar[["FID","IID","sex","age"]], on=["FID","IID"], how="left")
covar.to_csv("covar.tsv", sep="\t", index=False)

# 5) One-file version (optional)
merged = covar.merge(pheno[["FID","IID","meningitis"]], on=["FID","IID"], how="left")
merged.to_csv("merged.tsv", sep="\t", index=False)

# 6) Variant list
with open("variants.txt", "w") as fh:
    fh.write("rs17569141\n")

# 7) Quick sanity
print("WROTE: pheno.tsv, covar.tsv, merged.tsv, variants.txt")
print("N:", len(base),
      "| cases:", int((pheno['meningitis']==1).sum()),
      "| controls:", int((pheno['meningitis']==0).sum()),
      "| missing pheno:", int(pheno['meningitis'].isna().sum()))
print("\npheno.tsv (head):")
display(pd.read_csv("pheno.tsv", sep="\t").head())
print("\ncovar.tsv (head):")
display(pd.read_csv("covar.tsv", sep="\t").head())
print("\nmerged.tsv (head):")
display(pd.read_csv("merged.tsv", sep="\t").head())


In [ ]:
#GET SAMPLE PCs

In [ ]:
!gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv .

In [ ]:
ancestry_pred = pd.read_csv("ancestry_preds.tsv", delimiter="\t")
#ancestry_pred.to_csv('raw_ancestry_pcs.csv')

In [ ]:
ancestry_pred

In [ ]:
ancestry_pred["pca_features"] = ancestry_pred["pca_features"].str[1:-1]

In [ ]:
PCs = ancestry_pred["pca_features"].str.split(",", n = 16, expand = True)
PCs = PCs.astype(float)

In [ ]:
pid = ancestry_pred[["research_id"]]
pid.head(5)

In [ ]:
columns= ["PC1","PC2","PC3","PC4","PC5","PC6","PC7","PC8","PC9","PC10","PC11","PC12","PC13","PC14","PC15","PC16"]
PCs.columns = columns

In [ ]:
PCs.head(5)

In [ ]:
PCs_final = pd.concat([pid, PCs], axis = 1)
PCs_final.head(5)

In [ ]:
PCs_final.to_csv('wrangled_ancestry_pcs.csv')

In [ ]:
# Merge two phenotype file
pheno_final = pheno.merge(PCs_final, left_on = "person_id", right_on = "research_id", how = "left").drop(["research_id"], axis = 1)
pheno_final.head(5)

In [ ]:
#With PCs

import pandas as pd
import numpy as np
import re

# ======= EDIT THESE to match your data =======
ID_CASES     = "person_id"     # ID in cases_df
ID_CONTROLS  = "person_id"     # ID in controls_df
SEX_CASES    = "sex_at_birth"  # sex in cases_df (M/F or 1/2)
SEX_CONTROLS = "sex_at_birth"  # sex in controls_df (M/F or 1/2)

# PCs table (already loaded as PCs_final)
PC_ID_COL    = "research_id"   # ID column in PCs_final
PC_PREFIX    = "PC"            # PC columns start with something like PC1, pc_02, "PC 3", etc.
# ============================================

def normalize_pc_columns(pcs_df, pc_id_col, prefix="PC"):
    """Rename variants of PC columns to 'PC<k>' (pc1, PC_01, 'PC 2' -> PC1/PC2), keep only ID + PCs."""
    df = pcs_df.copy()
    # strip spaces around names
    df.columns = [str(c).strip() for c in df.columns]

    rename = {}
    for c in df.columns:
        if c == pc_id_col:
            continue
        s = str(c).strip()
        # match pc, PC, PC_, PC<space> and zero-padded numbers
        m = re.match(r'(?i)^pc[\s_]*0*([0-9]+)$', s)
        if not m:
            # also handle 'principal component 1' / 'pcscore1' styles if present
            m = re.match(r'(?i)^(principal[\s_]*component|pcscore)[\s_]*0*([0-9]+)$', s)
            if m:
                num = m.group(2)
                rename[c] = f'PC{int(num)}'
                continue
        if m:
            num = m.group(1)
            rename[c] = f'PC{int(num)}'
    df = df.rename(columns=rename)

    # keep ID + PC* columns only
    pc_cols = [c for c in df.columns if c != pc_id_col and str(c).upper().startswith(prefix.upper())]
    # ensure numeric PCs
    for c in pc_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # sort PC columns in numeric order (PC1, PC2, ..., PC10)
    def pc_num(name):
        m = re.search(r'(\d+)$', name)
        return int(m.group(1)) if m else 10**9

    pc_cols = sorted(pc_cols, key=pc_num)
    return df[[pc_id_col] + pc_cols], pc_cols

# ---------- 1) Force ID dtypes to string everywhere ----------
cases = cases.copy()
ctrls = ctrls.copy()
PCs_final = PCs_final.copy()

cases[ID_CASES] = cases[ID_CASES].astype("string")
ctrls[ID_CONTROLS] = ctrls[ID_CONTROLS].astype("string")
PCs_final[PC_ID_COL] = PCs_final[PC_ID_COL].astype("string")

# ---------- 2) Base sample list (union of IDs; FID=IID) ----------
case_ids = pd.Series(cases[ID_CASES].dropna().unique(), dtype="string", name="IID")
ctrl_ids = pd.Series(ctrls[ID_CONTROLS].dropna().unique(), dtype="string", name="IID")
base = pd.DataFrame({"IID": pd.concat([case_ids, ctrl_ids], ignore_index=True).drop_duplicates()})
base["FID"] = base["IID"]
base = base.astype({"FID":"string","IID":"string"})[["FID","IID"]]

# ---------- 3) Phenotype (1=case, 0=control, NA otherwise) ----------
pheno = base.copy()
case_set, ctrl_set = set(case_ids), set(ctrl_ids)
pheno["meningitis"] = np.where(
    pheno["IID"].isin(case_set), 1,
    np.where(pheno["IID"].isin(ctrl_set), 0, np.nan)
)
pheno.to_csv("pheno.tsv", sep="\t", index=False)

# ---------- 4) Covariates: sex, age + PCs from separate table ----------
dem_cases = cases[[ID_CASES, SEX_CASES, "age"]].rename(columns={ID_CASES:"IID", SEX_CASES:"sex"})
dem_ctrls = ctrls[[ID_CONTROLS, SEX_CONTROLS, "age"]].rename(columns={ID_CONTROLS:"IID", SEX_CONTROLS:"sex"})
for df in (dem_cases, dem_ctrls):
    df["IID"] = df["IID"].astype("string")

covar = pd.concat([dem_cases, dem_ctrls], ignore_index=True).drop_duplicates(subset=["IID"])
covar["FID"] = covar["IID"]
covar["sex"] = (covar["sex"].astype(str).str.strip().str.upper()
                .replace({"MALE":"M","FEMALE":"F","1":"M","2":"F","0":np.nan,"UNKNOWN":np.nan}))
covar["age"] = pd.to_numeric(covar["age"], errors="coerce")

# --- normalize & merge PCs ---
pcs_norm, pc_cols_found = normalize_pc_columns(PCs_final, PC_ID_COL, prefix=PC_PREFIX)
pcs_norm = pcs_norm.rename(columns={PC_ID_COL:"IID"})
pcs_norm["IID"] = pcs_norm["IID"].astype("string")

covar = covar.merge(pcs_norm, on="IID", how="left")

# order columns: FID IID sex age then PCs (numeric order)
def pc_key(name):
    m = re.search(r'(\d+)$', name)
    return (int(m.group(1)) if m else 10**9, name)
pc_order = sorted([c for c in covar.columns if c.upper().startswith(PC_PREFIX.upper())], key=pc_key)

covar = base.merge(covar[["FID","IID","sex","age"] + pc_order], on=["FID","IID"], how="left")
covar.to_csv("covar.tsv", sep="\t", index=False)

# ---------- 5) One-file version ----------
merged = covar.merge(pheno[["FID","IID","meningitis"]], on=["FID","IID"], how="left")
merged.to_csv("merged.tsv", sep="\t", index=False)

# ---------- 6) Variant list ----------
with open("variants.txt", "w") as fh:
    fh.write("rs17569141\n")

# ---------- 7) Diagnostics & previews ----------
try:
    print("Index date used for controls:", index_date.date())
except Exception:
    pass

print("PC columns detected & merged:", pc_cols_found if pc_cols_found else "None found")
# How many IDs overlapped between PCs and your base?
overlap_n = covar["IID"].notna().sum()
print(f"Samples in base: {len(base)}  | with any covariate row: {len(set(pd.concat([dem_cases['IID'], dem_ctrls['IID']])))}  | with PCs merged: {(~pcs_norm.drop(columns=['IID']).isna().all(axis=1)).sum()}")

print("\nWROTE: pheno.tsv, covar.tsv, merged.tsv, variants.txt")
print("N total:", len(base),
      "| cases:", int((pheno['meningitis']==1).sum()),
      "| controls:", int((pheno['meningitis']==0).sum()),
      "| missing pheno:", int(pheno['meningitis'].isna().sum()))

print("\npheno.tsv (head):")
display(pd.read_csv("pheno.tsv", sep="\t").head())

print("\ncovar.tsv (head):")
display(pd.read_csv("covar.tsv", sep="\t").head())

print("\nmerged.tsv (head):")
display(pd.read_csv("merged.tsv", sep="\t").head())


In [ ]:
merged.tail(10)